# Análisis de Sueldos vs. Costo de Vida en Chile (2022–2025)
**Autor:** Claudio Galiano · **Metodología:** CRISP-DM · **Herramientas:** Python (Google Colab) y Power BI

**Objetivo:** analizar la relación entre los ingresos laborales y el costo de vida en Chile, identificando diferencias por región, género, educación y sector económico, y sus posibles causas.

| Sección | Contenido | Estado |
|---|---|---|
| 1 | Configuración | Listo |
| 2 | Carga de datos | Listo |
| 3 | Comprensión de los datos (exploración) | Pendiente |
| 4 | Preparación de los datos | Pendiente |
| 5 | Modelo dimensional (cubo) | Pendiente |
| 6 | Análisis y minería de datos | Pendiente |
| 7 | Conclusiones | Pendiente |

## 1. Configuración
Antes de trabajar hay que preparar el "escritorio": conectar Google Drive (donde están los datos), cargar las herramientas de Python y decirle al notebook dónde están las carpetas.

### 1.1 Conectar Google Drive
**Qué hace:** le da permiso a Colab para leer los archivos de tu Drive. Al ejecutarla aparece una ventana: elige tu cuenta y acepta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Cargar librerías
**Qué hace:** las librerías son "cajas de herramientas" que otras personas ya programaron:
- **os** → maneja carpetas y rutas de archivos.
- **zipfile** → abre archivos comprimidos (.zip), como el de la CASEN.
- **numpy** → cálculos numéricos.
- **pandas** → trabaja con tablas de datos (como un Excel dentro de Python). Es la principal.
- **matplotlib / seaborn** → hacen gráficos.
- **pyreadstat** → permite leer archivos SPSS (.sav), el formato de la CASEN. No viene instalada en Colab, por eso se instala con `!pip install`.

In [ ]:
!pip install -q pyreadstat

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mostrar todas las columnas y los números con separador de miles
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Librerías cargadas')

### 1.3 Rutas del proyecto
**Qué hace:** guarda en variables la ubicación de las carpetas, para no escribirla completa cada vez. Luego revisa que los archivos estén donde deben.

> Si moviste la carpeta del proyecto en Drive, solo cambia la línea `RUTA`.

In [ ]:
RUTA = '/content/drive/MyDrive/Proyecto_Ingresos_Chile/'
CRUDOS = os.path.join(RUTA, '01_datos_crudos')

for carpeta in ['esi', 'ipc', 'casen']:
    archivos = os.listdir(os.path.join(CRUDOS, carpeta))
    print(f'{carpeta}: {archivos}')

## 2. Carga de datos
Aquí se **leen** los archivos originales y se guardan en tablas de pandas (llamadas *DataFrames*). Todavía **no se limpia ni se cambia nada**: solo se cargan para conocerlos en la sección 3.

| Tabla | Fuente | Qué contiene |
|---|---|---|
| `esi` | INE · Encuesta Suplementaria de Ingresos 2022–2025 | Una fila por persona: región, sexo, edad, educación, trabajo e **ingreso** |
| `ipc_general` | INE vía Banco Central | IPC mensual general (mide cuánto suben los precios) |
| `ipc_divisiones` | INE vía Banco Central | IPC mensual de 13 tipos de gasto (alimentos, vivienda, transporte…) |
| `ipc_ponderadores` | INE vía Banco Central | Cuánto pesa cada tipo de gasto en el presupuesto de los hogares |
| `casen` | Ministerio de Desarrollo Social · CASEN 2024 | Una fila por persona; incluye cuánto paga su hogar de **arriendo** |

### 2.1 ESI 2022–2025 (ingresos)
**Qué hace:**
1. Recorre los 4 años con un ciclo `for`.
2. Lee cada archivo CSV con `pd.read_csv`.
3. Le agrega una columna `anio` para saber de qué año es cada fila.
4. Une los 4 años en una sola tabla con `pd.concat`.

Tarda unos segundos porque cada archivo pesa ~80 MB.

In [ ]:
anios = [2022, 2023, 2024, 2025]
lista_esi = []

for anio in anios:
    archivo = os.path.join(CRUDOS, 'esi', f'esi_{anio}.csv')
    df_anio = pd.read_csv(archivo, low_memory=False)
    df_anio['anio'] = anio
    lista_esi.append(df_anio)
    print(f'ESI {anio}: {df_anio.shape[0]:,} filas · {df_anio.shape[1]} columnas')

esi = pd.concat(lista_esi, ignore_index=True)
print(f'\nESI total: {esi.shape[0]:,} filas · {esi.shape[1]} columnas')

**Primer vistazo:** `head()` muestra las primeras 5 filas. Verás que casi todo son **números**: muchos son **códigos** (por ejemplo, en `sexo`, 1 = Hombre y 2 = Mujer). Su significado está en los libros de códigos (`02_documentacion_fuentes`).

In [ ]:
esi.head()

> **Observación.** Los cuatro años suman 369.731 registros. La muestra de 2024 es la más chica (85.172 personas), así que conviene tenerlo presente al comparar ese año. El archivo de 2025 trae 290 columnas y los anteriores 284, porque el INE incorporó preguntas nuevas; al unirlos quedan 292 columnas y las nuevas aparecen vacías en los años que no las tenían. Casi todas las variables vienen como códigos numéricos, por lo que el libro de códigos es indispensable para interpretarlas.

### 2.2 IPC (precios)
**Qué hace:** lee 3 archivos Excel descargados del Banco Central. Todos tienen la hoja `Cuadro` y 2 filas de título arriba, por eso se usa `skiprows=2` (salta esas filas).

- **IPC general:** una fila por mes, de enero 2022 a diciembre 2025.
- **IPC por división:** una fila por mes y una columna por tipo de gasto (desde 2023).
- **Ponderadores:** el peso (%) de cada tipo de gasto en la canasta.

In [ ]:
ipc_general = pd.read_excel(os.path.join(CRUDOS, 'ipc', 'ipc_general_empalmado_2022_2025.xlsx'),
                            sheet_name='Cuadro', skiprows=2)
ipc_divisiones = pd.read_excel(os.path.join(CRUDOS, 'ipc', 'ipc_divisiones_2023_2025.xlsx'),
                               sheet_name='Cuadro', skiprows=2)
ipc_ponderadores = pd.read_excel(os.path.join(CRUDOS, 'ipc', 'ipc_ponderadores_divisiones_2023.xlsx'),
                                 sheet_name='Cuadro', skiprows=2)

print(f'IPC general:     {ipc_general.shape[0]} filas · {ipc_general.shape[1]} columnas')
print(f'IPC divisiones:  {ipc_divisiones.shape[0]} filas · {ipc_divisiones.shape[1]} columnas')
print(f'Ponderadores:    {ipc_ponderadores.shape[0]} fila · {ipc_ponderadores.shape[1]} columnas')
ipc_general.head()

> **Observación.** El IPC general cubre los 48 meses entre enero de 2022 y diciembre de 2025, sin vacíos. Las divisiones parten recién en 2023 porque ese año el INE cambió la canasta y pasó de 12 a 13 divisiones; mezclar ambas bases no sería correcto. Según los ponderadores, alimentos (22,2 %), vivienda (16,8 %) y transporte (13,5 %) explican más de la mitad del gasto de los hogares, así que cualquier alza en esos tres rubros se nota mucho más en el bolsillo.

### 2.3 CASEN 2024 (arriendo)
**Qué hace:**
1. Descomprime el archivo `.zip` (adentro viene `casen_2024.sav`, formato SPSS).
2. Lo lee con `pd.read_spss`, pero **solo las 7 columnas que necesitamos**: la base completa tiene cientos de columnas y podría colapsar Colab.

| Columna | Significado |
|---|---|
| `folio` | Número que identifica a cada hogar |
| `region` | Región (1 a 16, mismos códigos que la ESI) |
| `area` | Urbano o rural |
| `ten_viv` | Tenencia de la vivienda (2 = arrendada) |
| `v18` | Cuánto paga el hogar de arriendo (-88 = no sabe) |
| `v19` | Cuánto se paga de arriendo en el sector por viviendas parecidas |
| `expr` | Factor de expansión: a cuántas personas reales representa cada fila |

In [ ]:
zip_casen = os.path.join(CRUDOS, 'casen', 'casen_2024spss.zip')
carpeta_temporal = '/content/casen'
os.makedirs(carpeta_temporal, exist_ok=True)

with zipfile.ZipFile(zip_casen) as z:
    archivo_sav = [n for n in z.namelist() if n.lower().endswith('.sav')][0]
    z.extract(archivo_sav, carpeta_temporal)

columnas_casen = ['folio', 'region', 'area', 'ten_viv', 'v18', 'v19', 'expr']
casen = pd.read_spss(os.path.join(carpeta_temporal, archivo_sav),
                     usecols=columnas_casen, convert_categoricals=False)

print(f'CASEN 2024: {casen.shape[0]:,} personas · {casen["folio"].nunique():,} hogares')
casen.head()

> **Observación.** La CASEN viene con una fila por persona, pero el arriendo es un dato del hogar: si no se corrige, un hogar de cinco personas pesaría cinco veces. Por eso, antes de calcular, habrá que dejar una sola fila por cada folio. El monto de arriendo (v18) solo lo responden los hogares que arriendan, y el código -88 corresponde a "no sabe", que debe excluirse.

### 2.4 Resumen de lo cargado
**Qué hace:** junta en una tabla cuántas filas y columnas tiene cada fuente, para confirmar que todo se cargó.

In [ ]:
resumen = pd.DataFrame({
    'tabla': ['esi', 'ipc_general', 'ipc_divisiones', 'ipc_ponderadores', 'casen'],
    'filas': [len(esi), len(ipc_general), len(ipc_divisiones), len(ipc_ponderadores), len(casen)],
    'columnas': [esi.shape[1], ipc_general.shape[1], ipc_divisiones.shape[1],
                 ipc_ponderadores.shape[1], casen.shape[1]],
})
resumen

> **Observación general.** Las tres fuentes no tienen el mismo nivel de detalle: la ESI describe personas en un año, el IPC describe meses y la CASEN describe hogares. No se pueden unir fila por fila; la forma correcta de cruzarlas es a través de lo que tienen en común, la región y el tiempo. Esa es la base del modelo dimensional que se construirá más adelante.

---
**Siguiente:** sección 3, Comprensión de los datos.